# Fine-Tuning BERT for Multi-Class Emotion Detection

> **Dataset**: `dair-ai/emotion` — 6 emotion classes from English Twitter data  
> **Model**: `bert-base-uncased` fine-tuned for sequence classification  
> **Goal**: Classify text into one of: *sadness, joy, love, anger, fear, surprise*

---
## Outline
1. Setup & imports
2. Exploratory Data Analysis (EDA)
3. Tokenisation & data loading
4. Model architecture overview
5. Training
6. Evaluation & metrics
7. Error analysis
8. Inference demo

## 1. Setup

In [ ]:
# Install dependencies
# !pip install transformers datasets scikit-learn torch matplotlib seaborn wordcloud

In [ ]:
import os, json, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from wordcloud import WordCloud
from collections import Counter

import torch
from datasets import load_dataset
from transformers import BertTokenizerFast, BertForSequenceClassification
from sklearn.metrics import (
    accuracy_score, f1_score,
    classification_report, confusion_matrix,
)

warnings.filterwarnings('ignore')
plt.rcParams.update({'figure.dpi': 120, 'font.family': 'DejaVu Sans'})

LABELS   = ['sadness', 'joy', 'love', 'anger', 'fear', 'surprise']
PALETTE  = ['#3B82F6', '#F59E0B', '#EC4899', '#EF4444', '#8B5CF6', '#10B981']
LABEL_MAP = {i: l for i, l in enumerate(LABELS)}

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

## 2. Exploratory Data Analysis

In [ ]:
ds = load_dataset('dair-ai/emotion')
print(ds)

# Convert to DataFrames
dfs = {split: ds[split].to_pandas() for split in ds}
for split, df in dfs.items():
    df['emotion'] = df['label'].map(LABEL_MAP)
    print(f'{split:12s}: {len(df):,} rows')

In [ ]:
# ── Class distribution ────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 4), sharey=False)
fig.suptitle('Class Distribution by Split', fontsize=14, fontweight='bold')

for ax, (split, df) in zip(axes, dfs.items()):
    counts = df['emotion'].value_counts().reindex(LABELS)
    bars = ax.bar(LABELS, counts.values, color=PALETTE, width=0.6)
    ax.set_title(split.capitalize())
    ax.set_xlabel('Emotion')
    ax.set_ylabel('Count')
    ax.tick_params(axis='x', rotation=30)
    for bar in bars:
        ax.annotate(f'{bar.get_height():,}',
                    xy=(bar.get_x() + bar.get_width()/2, bar.get_height()),
                    xytext=(0, 4), textcoords='offset points',
                    ha='center', fontsize=9)

plt.tight_layout()
plt.show()

In [ ]:
# ── Text length distribution ──────────────────────────────────────────────────
train_df = dfs['train'].copy()
train_df['num_words'] = train_df['text'].str.split().apply(len)
train_df['num_chars'] = train_df['text'].str.len()

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
fig.suptitle('Training Set — Text Length Statistics', fontsize=13, fontweight='bold')

axes[0].hist(train_df['num_words'], bins=50, color='#3B82F6', edgecolor='white')
axes[0].axvline(train_df['num_words'].median(), color='red', ls='--', label=f'Median={train_df["num_words"].median():.0f}')
axes[0].set(title='Word count', xlabel='Words per sample', ylabel='Frequency')
axes[0].legend()

axes[1].hist(train_df['num_chars'], bins=50, color='#10B981', edgecolor='white')
axes[1].axvline(train_df['num_chars'].median(), color='red', ls='--', label=f'Median={train_df["num_chars"].median():.0f}')
axes[1].set(title='Character count', xlabel='Characters per sample', ylabel='Frequency')
axes[1].legend()

plt.tight_layout()
plt.show()

print(train_df[['num_words', 'num_chars']].describe().round(1))

In [ ]:
# ── Word clouds per emotion ───────────────────────────────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(16, 8))
fig.suptitle('Word Clouds by Emotion Class', fontsize=14, fontweight='bold')

for ax, (label, color) in zip(axes.flatten(), zip(LABELS, PALETTE)):
    corpus = ' '.join(train_df[train_df['emotion'] == label]['text'])
    wc = WordCloud(
        width=400, height=220, background_color='white',
        color_func=lambda *args, **kwargs: color,
        max_words=60, collocations=False,
    ).generate(corpus)
    ax.imshow(wc, interpolation='bilinear')
    ax.set_title(label.capitalize(), color=color, fontweight='bold')
    ax.axis('off')

plt.tight_layout()
plt.show()

## 3. Tokenisation

In [ ]:
tokenizer = BertTokenizerFast.from_pretrained('bert-base-uncased')

# Show tokenisation example
sample_text = 'I can\'t believe how happy I am right now!'
tokens = tokenizer(sample_text, return_tensors='pt')
print('Input text  :', sample_text)
print('Token IDs   :', tokens['input_ids'][0].tolist())
print('Tokens      :', tokenizer.convert_ids_to_tokens(tokens['input_ids'][0].tolist()))
print('Attention   :', tokens['attention_mask'][0].tolist())
print(f'Vocab size  : {tokenizer.vocab_size:,}')

In [ ]:
# Analyse token-length coverage at max_len=128
MAX_LEN = 128
sample_n = min(5000, len(train_df))
lengths = [
    len(tokenizer.encode(t, truncation=False))
    for t in train_df['text'].sample(sample_n, random_state=42)
]

coverage = sum(l <= MAX_LEN for l in lengths) / len(lengths)
print(f'Samples ≤ {MAX_LEN} tokens: {coverage:.1%}  (n={sample_n})')
print(f'Max token length in sample: {max(lengths)}')

plt.figure(figsize=(10, 4))
plt.hist(lengths, bins=40, color='#6366F1', edgecolor='white')
plt.axvline(MAX_LEN, color='red', ls='--', lw=2, label=f'max_len={MAX_LEN} ({coverage:.1%} coverage)')
plt.title('Token Length Distribution (BERT tokeniser)')
plt.xlabel('Token count'); plt.ylabel('Frequency')
plt.legend(); plt.tight_layout(); plt.show()

## 4. Model Architecture

In [ ]:
model = BertForSequenceClassification.from_pretrained(
    'bert-base-uncased',
    num_labels=6,
    id2label={i: l for i, l in enumerate(LABELS)},
    label2id={l: i for i, l in enumerate(LABELS)},
)

total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Total parameters      : {total_params:,}')
print(f'Trainable parameters  : {trainable_params:,}')
print(f'\nModel config:')
print(f'  Hidden size         : {model.config.hidden_size}')
print(f'  Attention heads     : {model.config.num_attention_heads}')
print(f'  Encoder layers      : {model.config.num_hidden_layers}')
print(f'  Max position embeddings: {model.config.max_position_embeddings}')
print(f'  Vocab size          : {model.config.vocab_size:,}')

## 5. Training

Run training from the CLI for best performance:
```bash
python src/train.py
```
Or run the cell below for a notebook-integrated run.

In [ ]:
# Optional: run training inline (slower — use CLI instead for production)
import subprocess
# subprocess.run(['python', 'src/train.py'], check=True)

## 6. Evaluation

In [ ]:
# Load training history
with open('./logs/results.json') as f:
    results = json.load(f)

history = results['history']
print(f"Best epoch : {results['best_epoch']}")
print(f"Test accuracy  : {results['accuracy']:.4f}")
print(f"Test F1 macro  : {results['f1_macro']:.4f}")
print(f"Test F1 weighted: {results['f1_weighted']:.4f}")

In [ ]:
# ── Training curves ───────────────────────────────────────────────────────────
ep = range(1, len(history['train_loss']) + 1)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('BERT Fine-tuning Curves', fontsize=14, fontweight='bold')

axes[0].plot(ep, history['train_loss'], 'o-', color='#2563EB', label='Train')
axes[0].plot(ep, history['val_loss'],   's--', color='#DC2626', label='Val')
axes[0].set(title='Cross-Entropy Loss', xlabel='Epoch', ylabel='Loss')
axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].plot(ep, history['train_f1'], 'o-', color='#2563EB', label='Train')
axes[1].plot(ep, history['val_f1'],   's--', color='#DC2626', label='Val')
axes[1].set(title='Macro F1 Score', xlabel='Epoch', ylabel='F1', ylim=[0, 1])
axes[1].legend(); axes[1].grid(alpha=0.3)

plt.tight_layout(); plt.show()

In [ ]:
# ── Full test-set evaluation ──────────────────────────────────────────────────
from torch.utils.data import DataLoader

def get_predictions(model, loader):
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for batch in loader:
            b = {k: v.to(device) for k, v in batch.items()}
            preds = model(**b).logits.argmax(-1).cpu().numpy()
            all_preds.extend(preds)
            all_labels.extend(b['labels'].cpu().numpy())
    return all_preds, all_labels

def tokenize_ds(ds_split):
    enc = tokenizer(
        list(ds_split['text']),
        padding='max_length', truncation=True, max_length=128, return_tensors='pt'
    )
    from torch.utils.data import TensorDataset
    return TensorDataset(
        enc['input_ids'], enc['attention_mask'],
        torch.tensor(list(ds_split['label']), dtype=torch.long)
    )

best_model = BertForSequenceClassification.from_pretrained('./models/bert-emotion').to(device)
test_ds    = tokenize_ds(ds['test'])

from torch.utils.data import DataLoader as DL, TensorDataset
class LabelDataset(torch.utils.data.Dataset):
    def __init__(self, input_ids, attention_mask, labels):
        self.input_ids      = input_ids
        self.attention_mask = attention_mask
        self.labels         = labels
    def __len__(self): return len(self.labels)
    def __getitem__(self, i):
        return {'input_ids': self.input_ids[i],
                'attention_mask': self.attention_mask[i],
                'labels': self.labels[i]}

enc       = tokenizer(list(ds['test']['text']), padding='max_length',
                      truncation=True, max_length=128, return_tensors='pt')
test_dset = LabelDataset(enc['input_ids'], enc['attention_mask'],
                         torch.tensor(list(ds['test']['label']), dtype=torch.long))
test_loader = DL(test_dset, batch_size=64)

preds, labels = get_predictions(best_model, test_loader)
print(classification_report(labels, preds, target_names=LABELS))

In [ ]:
# ── Confusion matrix ──────────────────────────────────────────────────────────
cm = confusion_matrix(labels, preds)
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=LABELS, yticklabels=LABELS, ax=axes[0])
axes[0].set(title='Confusion Matrix (counts)', xlabel='Predicted', ylabel='True')

sns.heatmap(cm_norm, annot=True, fmt='.2f', cmap='Blues',
            xticklabels=LABELS, yticklabels=LABELS, ax=axes[1], vmin=0, vmax=1)
axes[1].set(title='Confusion Matrix (normalised)', xlabel='Predicted', ylabel='True')

plt.tight_layout(); plt.show()

## 7. Error Analysis

In [ ]:
test_df = ds['test'].to_pandas()
test_df['pred']    = preds
test_df['true']    = labels
test_df['emotion_true'] = test_df['true'].map(LABEL_MAP)
test_df['emotion_pred'] = test_df['pred'].map(LABEL_MAP)
test_df['correct']      = test_df['pred'] == test_df['true']

errors = test_df[~test_df['correct']]
print(f'Total errors: {len(errors)} / {len(test_df)}  ({len(errors)/len(test_df):.1%})')

# Most common error pairs
error_pairs = errors.groupby(['emotion_true', 'emotion_pred']).size().reset_index(name='count')
error_pairs = error_pairs.sort_values('count', ascending=False).head(10)
print('\nTop misclassifications:')
print(error_pairs.to_string(index=False))

In [ ]:
# Sample hard examples
print('=== Sample Misclassified Examples ===\n')
for _, row in errors.sample(min(8, len(errors)), random_state=42).iterrows():
    print(f'TRUE: {row["emotion_true"]:10s} | PRED: {row["emotion_pred"]:10s}')
    print(f'TEXT: {row["text"][:120]}')
    print()

## 8. Inference Demo

In [ ]:
import sys; sys.path.insert(0, '.')
from src.predict import EmotionPredictor

predictor = EmotionPredictor('./models/bert-emotion')

demo_texts = [
    'I just found out I got into my dream university!',
    'The dog I grew up with just passed away. I miss him so much.',
    'I cannot believe you would betray me like that!',
    'Something is in the attic and it keeps making noises at night.',
    'You got a job at NASA?! I had no idea you even applied!',
    'You are the most important person in my world.',
]

print('─' * 65)
for text in demo_texts:
    r = predictor.predict(text)
    bar  = '█' * int(r.confidence * 30)
    print(f'{r.emoji}  {r.label.upper():10s}  {r.confidence:.1%}  {bar}')
    print(f'   {text[:70]}')
    print()

In [ ]:
# Interactive: type your own text
user_text = 'I stayed up all night worrying about the presentation.'  # ← change me
result = predictor.predict(user_text)
print(result)